# 1-halo tSZ power spectrum, summed from the catalogue

We predict the thermal SZ angular power spectrum **directly from the cluster catalogue** as a
Poisson (1-halo / shot-noise) sum over the GNFW pressure profile of each halo, and compare it to the
NaMaster map bandpowers stored in `data/`. The 2-halo (clustering) term is set aside.

For a full-sky catalogue the 1-halo term is simply the shot power of the per-halo harmonic profiles,
$$
C_\ell^{1h}=\frac{1}{4\pi}\sum_i \big|y_\ell(M_i,z_i)\big|^2 .
$$
Each halo's harmonic Compton-$y$ profile factorises into an **amplitude** and a **shape**,
$$
y_\ell(M,z)=Y_{\rm ang}\,\hat G(\ell\,\theta_{500}),\qquad
Y_{\rm ang}=\frac{Y_{5R500c}}{D_A^2},\qquad \theta_{500}=\frac{R_{500c}}{D_A},
$$
where $Y_{\rm ang}$ is the integrated Compton-$Y$ in **steradians** and $\theta_{500}$ is the proper
angular scale. Because we **assume the A10 GNFW shape with $B=1$**, all of $P_0,c_{500},\alpha,\beta,
\gamma$ are *fixed*: the form factor $\hat G$ is then a **single universal curve** (independent of mass
and redshift). The Arnaud profile is calibrated directly at $500c$, so we use the catalogue $M_{500c}$,
$R_{500c}$ with no mass-definition conversion. We take the shape parameters from hmfast's
`GNFWPressureProfile` and the officially-provided catalogue $Y_{5R500c}$ for the amplitude, so this is a
genuine sum over real halos.

The catalogue already stores the proper angular size `theta500_arcmin`, so
$Y_{\rm ang}=Y_{5R500c}(\theta_{500}/R_{500c})^2$ needs no extra distance.

**SNR thresholds.** The masked maps remove discs around clusters detected above $q_{\rm cut}$, so each
masked spectrum is sourced by the survivors $q<q_{\rm cut}$; we sum over $q<q_{\rm cut}$ for
$q_{\rm cut}\in\{50,20,10,5\}$.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})


from flamingo import paths
from flamingo.catalogue import load_catalogue
from hmfast.halos.profiles import GNFWPressureProfile

ARCMIN_PER_RAD = 180.0 / np.pi * 60.0

# ---- catalogue: officially-provided Y_5R500c, proper theta500, SNR q ----
df = load_catalogue(paths.HYDRO / 'catalogue' / 'halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv')
df = df[(df['Y_5R500c_Mpc2'] > 0) & (df['R_500c_Mpc'] > 0) & np.isfinite(df['q'])]
R500 = df['R_500c_Mpc'].values                         # proper Mpc
Ympc2 = df['Y_5R500c_Mpc2'].values                     # = D_A^2 Y_sph, proper Mpc^2
qv = df['q'].values
th500 = df['theta500_arcmin'].values / ARCMIN_PER_RAD  # proper, radians
Yang = Ympc2 * (th500 / R500) ** 2                     # = Y_5R500c / D_A^2  [steradians]
print(f'clusters: {len(df)}')
print(f'white plateau  C_white = sum(Y_ang^2)/4pi = {np.sum(Yang**2)/(4*np.pi):.3e}  [sr]')

# ---- NaMaster map datapoints (data/) ----
dat = np.load(paths.DATA / 'nb09_tsz_map_ps.npz')
ellb = dat['ellb']                                     # bandpower centres
dl_map = dat['dl_map']                                 # full-sky D_ell
Q_CUTS = [50, 20, 10, 5]
dl_cut = {qc: dat[f'dl_q{qc}'] for qc in Q_CUTS}       # masked-map D_ell
print('map bandpowers:', f'{ellb.min():.0f} -> {ellb.max():.0f}')


clusters: 1555542
white plateau  C_white = sum(Y_ang^2)/4pi = 2.943e-16  [sr]
map bandpowers: 16 -> 5956


## Universal GNFW form factor

The Arnaud GNFW profile is calibrated **at $500c$**, and the catalogue gives $M_{500c}$, $R_{500c}$
directly, so we stay at $500c$ with no mass conversion. With the shape fixed (parameters taken from
hmfast's `GNFWPressureProfile`), the form factor is the normalised harmonic (Komatsu-Seljak / Limber)
transform of the 3D shape, truncated at the $5R_{500c}$ aperture,
$$
\hat G(u)=\frac{1}{I_5}\int_0^5 dx\,x^2\,p(x)\,\frac{\sin(ux)}{ux},\qquad u=\ell\,\theta_{500},\quad
I_5=\int_0^5 dx\,x^2 p(x),
$$
a single universal curve with $\hat G(0)=1$. The per-halo angular scale is the catalogue's proper
$\theta_{500}=R_{500c}/D_A$ (`theta500_arcmin`).


In [2]:
from scipy.integrate import quad

prof = GNFWPressureProfile(B=1.0)                      # fixed A10 shape, calibrated at 500c
P0, c500, al, be, ga = prof.P0, prof.c500, prof.alpha, prof.beta, prof.gamma
print(f'fixed A10 GNFW (500c): P0={P0}, c500={c500}, alpha={al}, beta={be}, gamma={ga}, B={prof.B}')

def p_shape(x):
    cx = c500 * np.maximum(x, 1e-12)
    return cx ** (-ga) * (1 + cx ** al) ** ((ga - be) / al)

I5 = quad(lambda x: p_shape(x) * x * x, 0, 5, limit=400)[0]
u_tab = np.geomspace(1e-3, 400.0, 600)
G_tab = np.array([quad(lambda x: p_shape(x) * x * x * np.sinc(u * x / np.pi), 0, 5, limit=400)[0] / I5
                  for u in u_tab])

def Ghat(u):
    return np.interp(np.clip(u, u_tab[0], u_tab[-1]), u_tab, G_tab)

def cl_1h(ell_arr, mask=None):
    w2 = (Yang if mask is None else Yang[mask]) ** 2
    t = th500 if mask is None else th500[mask]
    return np.array([np.sum(w2 * Ghat(l * t) ** 2) / (4 * np.pi) for l in ell_arr])

sel = (ellb > 400) & (ellb < 6000)
print(f'Ghat(1)={Ghat(1.0):.3f}  Ghat(3)={Ghat(3.0):.3f}  Ghat(6)={Ghat(6.0):.3f}')


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


fixed A10 GNFW (500c): P0=8.13, c500=1.156, alpha=1.062, beta=5.4807, gamma=0.3292, B=1.0


Ghat(1)=0.733  Ghat(3)=0.294  Ghat(6)=0.094


## Sum and compare

$D_\ell=\ell(\ell+1)C_\ell/2\pi$. The model curve is extended to very low $\ell$ to show the
1-halo white plateau ($C_\ell\to$ const, hence $D_\ell\propto\ell^2$). Full sky uses all halos; each
$q_{\rm cut}$ uses the survivors $q<q_{\rm cut}$.


In [3]:
def to_dl(ell_arr, cl):
    return ell_arr * (ell_arr + 1) * cl / (2 * np.pi)

def c_white(mask=None):                                # ell->0 Poisson plateau
    w = Yang if mask is None else Yang[mask]
    return np.sum(w ** 2) / (4 * np.pi)

ell_fine = np.geomspace(5.0, 8000.0, 70)               # extend to very low ell

cw_full = c_white()
cw_cut = {qc: c_white(qv < qc) for qc in Q_CUTS}

dl_full = to_dl(ell_fine, cl_1h(ell_fine))
dl_full_b = to_dl(ellb, cl_1h(ellb))
r = dl_full_b / dl_map
print(f'FULL SKY  median(cat-sum / map), 400<l<6000 = {np.median(r[sel]):.3f}')

dl_cut_fine, dl_cut_b = {}, {}
for qc in Q_CUTS:
    surv = qv < qc
    dl_cut_fine[qc] = to_dl(ell_fine, cl_1h(ell_fine, mask=surv))
    dl_cut_b[qc] = to_dl(ellb, cl_1h(ellb, mask=surv))
    rr = dl_cut_b[qc] / dl_cut[qc]
    print(f'q<{qc:<3d} ({int(surv.sum()):>7d} survivors)  '
          f'median(cat-sum / map) = {np.median(rr[sel]):.3f}')


FULL SKY  median(cat-sum / map), 400<l<6000 = 1.028


q<50  (1555540 survivors)  median(cat-sum / map) = 1.028


q<20  (1555450 survivors)  median(cat-sum / map) = 1.032


q<10  (1555006 survivors)  median(cat-sum / map) = 1.038


q<5   (1553150 survivors)  median(cat-sum / map) = 1.053


## Figure

In [4]:
colors = plt.cm.viridis(np.linspace(0.04, 0.85, len(Q_CUTS) + 1))
fig, (ax, axr) = plt.subplots(2, 1, figsize=(7.8, 7.6), sharex=True,
                              gridspec_kw={'height_ratios': [3, 1.3]})

ax.plot(ell_fine, dl_full, color=colors[0], lw=2.2, label='catalogue sum (all halos)')
ax.plot(ellb, dl_map, 'o', color=colors[0], ms=3.5, mfc='none', label='map (full sky)')
ax.plot(ell_fine, to_dl(ell_fine, cw_full), ':', color=colors[0], lw=1.3,
        label=r'white plateau $\ell\!\to\!0$  ($\sum Y_{\rm ang}^2/4\pi$)')
axr.plot(ellb, dl_full_b / dl_map, color=colors[0], lw=1.8)

for i, qc in enumerate(Q_CUTS):
    c = colors[i + 1]
    ax.plot(ell_fine, dl_cut_fine[qc], color=c, lw=2.0, label=fr'cat sum, $q<{qc}$')
    ax.plot(ellb, dl_cut[qc], 'o', color=c, ms=3.5, mfc='none', label=fr'map, mask $q>{qc}$')
    ax.plot(ell_fine, to_dl(ell_fine, cw_cut[qc]), ':', color=c, lw=1.3)
    axr.plot(ellb, dl_cut_b[qc] / dl_cut[qc], color=c, lw=1.8)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_ylabel(r'$D_\ell^{yy}=\ell(\ell+1)C_\ell/2\pi$', fontsize=12)
ax.set_title('1-halo tSZ power spectrum: catalogue sum (GNFW, $B=1$) vs map', fontsize=12)
ax.legend(fontsize=8, ncol=2, loc='upper left')
ax.tick_params(labelsize=10); ax.set_ylim(1e-16, 4e-12)

axr.axhline(1.0, color='k', lw=0.9, ls='--')
axr.fill_between([ellb.min(), ellb.max()], 0.9, 1.1, color='0.85', alpha=0.6)
axr.set_xlabel(r'multipole $\ell$', fontsize=12)
axr.set_ylabel('cat / map', fontsize=11)
axr.set_ylim(0.5, 1.4); axr.tick_params(labelsize=10)
fig.tight_layout()
fig.savefig('../autoresearch/figures/nb14_cat_sum_tsz_ps.pdf')
fig.savefig('../autoresearch/figures/nb14_cat_sum_tsz_ps.png', dpi=300)
plt.show()


## Notes

- **1-halo only.** The deficit at low $\ell$ ($\ell\lesssim300$) is the omitted 2-halo (clustering) term;
  at high $\ell$ the spectrum is profile/Poisson dominated and the catalogue sum tracks the map.
- The model curve continues to **very low $\ell$**, where the 1-halo term is the flat white plateau
  $C_\ell\to\frac{1}{4\pi}\sum_i Y_{{\rm ang},i}^2$, i.e. $D_\ell\propto\ell^2$.
- Amplitude uses the **real catalogue $Y_{5R500c}$** per halo; the only assumption is the fixed A10 GNFW
  **shape** ($B=1$), which sets the universal form factor $\hat G$.
- Threshold spectra use the simple survivor selection $q<q_{\rm cut}$; the masks also remove a little
  sky and some neighbours inside the discs, so exact agreement on the cuts is not expected.
